# Campus Help Desk Chatbot + Azure AI Language

Project 2: connecting a traditional chatbot to an AI-as-a-service.

* **Origin:** the intent classifier is copied from my Project 1 notebook (`TraditionalChatbot.ipynb`) so it can be extended rather than rebuilt.
* **New:** Azure AI Language sentiment analysis, language detection and key phrase extraction; a `help` capabilities list; malformed-input handling; graceful fallback when Azure is unavailable.

Before running: add two Colab secrets (key icon in the left sidebar): `MicrosoftAPIKey` and `MicrosoftAIServiceEndpoint`, both with notebook access turned on.

In [ ]:
# Step 1: install libraries
!pip install -q azure-ai-textanalytics gradio nltk scikit-learn

In [ ]:
# Step 2: get the project code from GitHub
!git clone -q https://github.com/phaneendra520/MSAI-Develop.git
%cd /content/MSAI-Develop/AzureChatbot
!ls

In [ ]:
# Step 3: load the Azure key and endpoint from Colab Secrets into environment variables
# (the key is never written into the notebook or the GitHub repository)
import os
from google.colab import userdata

os.environ['MicrosoftAPIKey'] = userdata.get('MicrosoftAPIKey')
os.environ['MicrosoftAIServiceEndpoint'] = userdata.get('MicrosoftAIServiceEndpoint')

print('Endpoint:', os.environ['MicrosoftAIServiceEndpoint'])   # debug only
print('Key loaded:', len(os.environ['MicrosoftAPIKey']) > 0)    # never print the key itself

In [ ]:
# Step 4: run the offline unit tests (Azure is simulated, so these pass without a key)
!python -m unittest test_chatbot.py -v

In [ ]:
# Step 5: create the bot and confirm the live Azure connection
from chatbot import Chatbot
bot = Chatbot()
print('Azure configured:', bot.azure.available)
print(bot.azure.analyze('I love how fast the help desk answered my question!'))
print('Last Azure error:', bot.azure.last_error)

In [ ]:
# Step 6: test multiple prompts, the capabilities list, sentiment and malformed input
tests = [
    'hi',
    'what can you do',
    'I forgot my password',
    'I forgot my password again and I am really frustrated!',
    'my wifi keeps dropping and it is driving me crazy',
    'what are the library hours',
    'thanks, that was great',
    'what is the weather today',
    'this bot is useless and I hate it',
    'olvidé mi contraseña, ¿me puedes ayudar?',
    '',
    '12345 !!! ???',
    'asdfgh qwrtz',
    'x' * 600,
]
for q in tests:
    shown = q if len(q) < 60 else q[:40] + '... (' + str(len(q)) + ' chars)'
    print(f'You: {shown!r}')
    print(f'Bot: {bot.reply(q)}\n')

In [ ]:
# Step 7: the analyze command (sentiment + language + key phrases)
print(bot.reply('analyze: The new library hours are fantastic, but the Wi-Fi on the third floor is terrible.'))

In [ ]:
# Step 8: launch the web chat interface
import gradio as gr
gr.ChatInterface(
    fn=lambda message, history: bot.reply(message),
    title='Campus Help Desk Bot + Azure AI Language',
    description="Type 'help' to see what I can do.",
).launch(share=True)